# Data Assembly & Feature Engineering — Cádiz Solar Forecast

This notebook assembles the raw data into a single dataset suitable for TFT modelling.

**Steps:**
1. Load ERA5-Land weather data for each of the 8 PV plants
2. Compute solar position (zenith, azimuth) and clear-sky irradiance (GHI, DNI, DHI) via `pvlib`
3. Create cyclical calendar features (hour sin/cos, month sin/cos)
4. Capacity-weighted aggregation across plants → single regional weather signature
5. Merge with ESIOS regional PV generation target
6. Export the assembled dataset

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import pvlib
from pvlib.location import Location

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## 1. Plant metadata

The 8 largest PV plants in the province of Cádiz with their GPS coordinates and installed capacity.

In [ ]:
PLANTS = {
    'Puerto_Real':     {'lat': 36.5152, 'lon': -6.2765, 'capacity_mwp': 230.50},
    'Amazon_Arco':     {'lat': 36.6902, 'lon': -6.1323, 'capacity_mwp': 137.50},
    'Las_Quinientas':  {'lat': 36.6123, 'lon': -6.1122, 'capacity_mwp': 110.00},
    'Miramundo':       {'lat': 36.5200, 'lon': -6.1310, 'capacity_mwp':  50.00},
    'Cartuja':         {'lat': 36.5806, 'lon': -6.1079, 'capacity_mwp':  46.80},
    'El_Yarte':        {'lat': 36.6061, 'lon': -5.7988, 'capacity_mwp':  46.60},
    'La_Guita':        {'lat': 36.6061, 'lon': -5.7988, 'capacity_mwp':  46.60},
    'Arenosas':        {'lat': 36.6061, 'lon': -5.7988, 'capacity_mwp':  46.60},
}

# Mapping from plant name -> ERA5 folder name (as they appear on disk)
PLANT_TO_FOLDER = {
    'Puerto_Real':     'ERA5-LAND_Cadiz_Puerto_Real_2023-2025',
    'Amazon_Arco':     'ERA5-LAND_Cadiz_Amazon_Arco_2023-2025',
    'Las_Quinientas':  'ERA5-LAND_Cadiz_Las_Quinientas_2023-2025',
    'Miramundo':       'ERA5-LAND_Cadiz_Miramundo_2023-2025',
    'Cartuja':         'ERA5-LAND_Cadiz_Cartuja_2023-2025',
    'El_Yarte':        'ERA5-LAND_Cadiz_El_Yarte_2023-2025',
    'La_Guita':        'ERA5-LAND_Cadiz_La_Guita_2023-2025',
    'Arenosas':        'ERA5-LAND_Cadiz_Arenosas_2023-2025',
}

DATA_DIR = 'data'

total_capacity = sum(p['capacity_mwp'] for p in PLANTS.values())
print(f'Total installed capacity: {total_capacity:.2f} MWp across {len(PLANTS)} plants')
for name, info in PLANTS.items():
    pct = info['capacity_mwp'] / total_capacity * 100
    print(f'  {name:25s}  {info["capacity_mwp"]:7.2f} MWp  ({pct:5.1f}%)  @ ({info["lat"]:.3f}, {info["lon"]:.3f})')

## 2. Load ERA5-Land weather data

For each plant we load the three ERA5-Land CSV files (temperature, pressure/precipitation, radiation) and merge them on timestamp.

**Variables loaded and converted:**
- `t2m` -> 2m air temperature (K -> C)
- `d2m` -> 2m dewpoint temperature (K -> C)
- `sp` -> Surface pressure (Pa -> hPa)
- `tp` -> Total precipitation (m -> mm)
- `ssrd` -> Surface solar radiation downwards (J/m2 per hour -> W/m2)
- `strd` -> Surface thermal radiation downwards (J/m2 per hour -> W/m2)

In [ ]:
def load_era5_plant(plant_name: str) -> pd.DataFrame:
    """Load and merge the 3 ERA5-Land CSVs for a given plant. Returns a DataFrame indexed by UTC datetime."""
    folder = os.path.join(DATA_DIR, PLANT_TO_FOLDER[plant_name])
    csvs = sorted(glob.glob(os.path.join(folder, '*.csv')))
    assert len(csvs) == 3, f'Expected 3 CSVs in {folder}, found {len(csvs)}'

    dfs = []
    for csv_path in csvs:
        df = pd.read_csv(csv_path, parse_dates=['valid_time'])
        df = df.set_index('valid_time').drop(columns=['latitude', 'longitude'])
        dfs.append(df)

    merged = pd.concat(dfs, axis=1)

    # Unit conversions
    merged['t2m'] = merged['t2m'] - 273.15          # K -> C
    merged['d2m'] = merged['d2m'] - 273.15          # K -> C
    merged['sp']  = merged['sp'] / 100.0            # Pa -> hPa
    merged['tp']  = merged['tp'] * 1000.0           # m -> mm
    merged['ssrd'] = merged['ssrd'] / 3600.0        # J/m2 (hourly) -> W/m2
    merged['strd'] = merged['strd'] / 3600.0        # J/m2 (hourly) -> W/m2

    # Rename for clarity
    merged = merged.rename(columns={
        't2m':  'temperature_2m_C',
        'd2m':  'dewpoint_2m_C',
        'sp':   'surface_pressure_hPa',
        'tp':   'total_precip_mm',
        'ssrd': 'ssrd_wm2',
        'strd': 'strd_wm2',
    })

    merged.index.name = 'datetime_utc'
    return merged


# Load all plants
era5_data = {}
for plant_name in PLANTS:
    era5_data[plant_name] = load_era5_plant(plant_name)
    print(f'{plant_name:25s}  {len(era5_data[plant_name]):6d} rows  '
          f'{era5_data[plant_name].index.min()} -> {era5_data[plant_name].index.max()}')

In [ ]:
# Quick sanity check
era5_data['Puerto_Real'].head(10)

## 3. Compute solar position & clear-sky irradiance (pvlib)

For each plant, using its exact GPS coordinates, we compute:
- **Solar zenith angle** (degrees) — angle between the sun and vertical
- **Solar azimuth angle** (degrees) — compass direction of the sun
- **Clear-sky GHI, DNI, DHI** (W/m2) — theoretical irradiance under cloudless conditions using the Ineichen model with climatological Linke turbidity

These are computed at the same UTC timestamps as the ERA5 data.

In [ ]:
def compute_solar_features(plant_name: str, timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    """
    Compute solar position and clear-sky irradiance for a plant at given UTC timestamps.

    Returns DataFrame with columns:
        solar_zenith, solar_azimuth, clearsky_ghi, clearsky_dni, clearsky_dhi
    """
    info = PLANTS[plant_name]
    loc = Location(latitude=info['lat'], longitude=info['lon'], tz='UTC', altitude=50)
    # altitude ~50m is a reasonable average for the Cádiz region

    # Ensure timestamps are UTC-localized
    if timestamps.tz is None:
        timestamps = timestamps.tz_localize('UTC')

    # Solar position
    solpos = loc.get_solarposition(timestamps)

    # Clear-sky irradiance (Ineichen model with climatological Linke turbidity)
    clearsky = loc.get_clearsky(timestamps, model='ineichen')

    result = pd.DataFrame({
        'solar_zenith':  solpos['apparent_zenith'].values,
        'solar_azimuth': solpos['azimuth'].values,
        'clearsky_ghi':  clearsky['ghi'].values,
        'clearsky_dni':  clearsky['dni'].values,
        'clearsky_dhi':  clearsky['dhi'].values,
    }, index=timestamps.tz_localize(None))  # store as tz-naive UTC

    result.index.name = 'datetime_utc'
    return result


# Compute for all plants
solar_data = {}
for plant_name in PLANTS:
    timestamps = era5_data[plant_name].index
    solar_data[plant_name] = compute_solar_features(plant_name, timestamps)
    cs_ghi_max = solar_data[plant_name]['clearsky_ghi'].max()
    print(f'{plant_name:25s}  max clear-sky GHI = {cs_ghi_max:.1f} W/m2')

solar_data['Puerto_Real'].head(15)

## 4. Merge weather + solar features per plant

Combine the ERA5 weather data and pvlib solar features into a single DataFrame per plant.

In [ ]:
plant_datasets = {}
for plant_name in PLANTS:
    combined = era5_data[plant_name].join(solar_data[plant_name], how='inner')
    plant_datasets[plant_name] = combined
    print(f'{plant_name:25s}  {len(combined)} rows, {combined.columns.tolist()}')

plant_datasets['Puerto_Real'].head()

## 5. Capacity-weighted aggregation -> regional weather signature

Since our target variable is regional PV generation for all of Cádiz, we aggregate the plant-level weather/solar features into a single set of regional features using **capacity-weighted averaging**.

Each plant's contribution is proportional to its installed capacity (MWp), so larger plants have more influence on the regional signature.

In [ ]:
# Trim all plant datasets to the common time range (2023-01-01 to 2025-12-31)
common_start = '2023-01-01'
common_end   = '2025-12-31 23:00:00'

for plant_name in PLANTS:
    df = plant_datasets[plant_name]
    plant_datasets[plant_name] = df.loc[common_start:common_end]

# Verify all have the same index
ref_index = plant_datasets['Puerto_Real'].index
for plant_name, df in plant_datasets.items():
    assert df.index.equals(ref_index), f'{plant_name} index mismatch!'
print(f'All plants aligned: {len(ref_index)} timestamps from {ref_index[0]} to {ref_index[-1]}')

In [ ]:
# Compute capacity weights (sum to 1)
total_cap = sum(p['capacity_mwp'] for p in PLANTS.values())
weights = {name: info['capacity_mwp'] / total_cap for name, info in PLANTS.items()}

print('Capacity weights:')
for name, w in weights.items():
    print(f'  {name:25s}  {w:.4f}  ({w*100:.1f}%)')
print(f'  Sum: {sum(weights.values()):.6f}')

In [ ]:
# Capacity-weighted average
feature_cols = plant_datasets['Puerto_Real'].columns.tolist()

regional_weather = pd.DataFrame(0.0, index=ref_index, columns=feature_cols)
for plant_name, w in weights.items():
    regional_weather += plant_datasets[plant_name][feature_cols] * w

regional_weather.index.name = 'datetime_utc'
print(f'Regional weather signature: {regional_weather.shape}')
regional_weather.describe()

## 6. Cyclical calendar features

We encode temporal information as cyclical sine/cosine transforms so that e.g. December and January appear close together rather than 11 months apart.

- **Hour of day** -> `hour_sin`, `hour_cos` (period = 24)
- **Month of year** -> `month_sin`, `month_cos` (period = 12)

In [ ]:
timestamps = regional_weather.index

# Hour of day (0-23)
hour = timestamps.hour
regional_weather['hour_sin'] = np.sin(2 * np.pi * hour / 24)
regional_weather['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Month of year (1-12)
month = timestamps.month
regional_weather['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
regional_weather['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

print('Calendar features added.')
regional_weather[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']].describe()

## 7. Load and merge ESIOS target variable (regional PV generation)

The ESIOS data is Cádiz regional solar generation downloaded from the ESIOS/REE platform.

The ESIOS timestamps are in CET/CEST (Europe/Madrid), while ERA5 is UTC. We convert ESIOS to UTC before merging. We use a **left join** on the weather data to preserve all timestamps — any hours missing from ESIOS will appear as NaN, which we can investigate during exploratory analysis.

In [ ]:
esios_file = glob.glob(os.path.join(DATA_DIR, 'ESIOS*.csv'))
assert len(esios_file) == 1, f'Expected 1 ESIOS file, found {len(esios_file)}'

esios = pd.read_csv(esios_file[0], sep=';')
esios = esios[['datetime', 'value']].rename(columns={'value': 'pv_generation_gwh'})

# Parse the ISO datetime strings (which include timezone offsets like +01:00 / +02:00)
# and convert to UTC
esios['datetime_utc'] = pd.to_datetime(esios['datetime'], utc=True).dt.tz_localize(None)
esios = esios.set_index('datetime_utc').drop(columns=['datetime'])
esios = esios.sort_index()

print(f'ESIOS data: {len(esios)} rows from {esios.index[0]} to {esios.index[-1]}')
print(f'Generation range: {esios["pv_generation_gwh"].min():.3f} -- {esios["pv_generation_gwh"].max():.3f} GWh')
esios.head(10)

In [ ]:
# Merge weather features with generation target (left join to keep all weather timestamps)
dataset = regional_weather.join(esios, how='left')

n_total = len(dataset)
n_missing_target = dataset['pv_generation_gwh'].isna().sum()
n_present = n_total - n_missing_target

print(f'Final merged dataset: {n_total} rows x {dataset.shape[1]} columns')
print(f'Date range: {dataset.index[0]} to {dataset.index[-1]}')
print(f'\nESIOS target coverage: {n_present}/{n_total} hours present ({n_present/n_total*100:.1f}%)')
print(f'Missing target hours: {n_missing_target} ({n_missing_target/n_total*100:.1f}%) -- to investigate in EDA')
print(f'\nColumns ({len(dataset.columns)}):')
for col in dataset.columns:
    print(f'  {col}')

dataset.head()

## 8. Quick data quality check

In [ ]:
print('=== Missing values ===')
print(dataset.isnull().sum())
print(f'\nTotal missing: {dataset.isnull().sum().sum()}')

print('\n=== Data types ===')
print(dataset.dtypes)

print('\n=== Descriptive statistics ===')
dataset.describe()

## 9. Export assembled dataset

In [ ]:
output_path = os.path.join(DATA_DIR, 'cadiz_assembled_dataset.csv')
dataset.to_csv(output_path)
print(f'Saved assembled dataset to: {output_path}')
print(f'Shape: {dataset.shape}')

## Summary

| Category | Features | Source |
|---|---|---|
| **Meteorological** | temperature_2m_C, dewpoint_2m_C, surface_pressure_hPa, total_precip_mm, ssrd_wm2, strd_wm2 | ERA5-Land (capacity-weighted avg) |
| **Solar position** | solar_zenith, solar_azimuth | pvlib (capacity-weighted avg) |
| **Clear-sky irradiance** | clearsky_ghi, clearsky_dni, clearsky_dhi | pvlib Ineichen model (capacity-weighted avg) |
| **Calendar (cyclical)** | hour_sin, hour_cos, month_sin, month_cos | Derived from timestamp |
| **Target** | pv_generation_gwh | ESIOS — Cádiz regional solar generation |

**Total:** 15 independent variables + 1 target = 16 columns

